# 🏛️ Système Multi-Agents — Génération de Contrats Juridiques

| Cellule | Rôle | Fréquence |
|---|---|---|
| 1 | Installation des dépendances | **Une seule fois** |
| 2 | Montage Google Drive + Config | À chaque session |
| 3 | Agent 1 — RAG Juridique | À chaque session |
| 4 | Agent 2 — Générateur de contrat | À chaque session |
| 5 | Orchestrateur | À chaque session |
| **6** | **▶️ Votre demande** | **Autant de fois que souhaité** |

> ⚠️ **Exécutez les cellules dans l'ordre de haut en bas.**

!pip install -q sentence-transformers faiss-cpu pypdf python-docx \
             llama-cpp-python huggingface_hub unicodedata2
print('✅ Installation terminée.')

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════
# ⚙️  MODIFIEZ CES CHEMINS SELON VOTRE GOOGLE DRIVE
# ══════════════════════════════════════════════════════
RAG_FOLDER    = '/content/drive/MyDrive/3D_SMART/lois'
MODELS_FOLDER = '/content/drive/MyDrive/3D_SMART/examples_des_contrats'
OUTPUT_FOLDER = '/content/drive/MyDrive/3D_Smart/Modèles de formulations pour tous les contrats/contrats_generes/'
# ══════════════════════════════════════════════════════

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'📂 RAG     : {RAG_FOLDER}')
print(f'📂 Modèles : {MODELS_FOLDER}')
print(f'💾 Sortie  : {OUTPUT_FOLDER}')

Mounted at /content/drive
📂 RAG     : /content/drive/MyDrive/3D_SMART/lois
📂 Modèles : /content/drive/MyDrive/3D_SMART/examples_des_contrats
💾 Sortie  : /content/drive/MyDrive/3D_Smart/Modèles de formulations pour tous les contrats/contrats_generes/


In [4]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 27.4 MB/s eta 0:00:0000:010:01m


In [5]:
import datetime
import os
import re
import fitz  # PyMuPDF
import numpy as np
from pathlib import Path
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from google.colab import drive

# ── Connexion Drive ──────────────────────────────────────────
drive.mount('/content/drive')

RAG_FOLDER    = '/content/drive/MyDrive/3D_SMART/lois'
OUTPUT_FOLDER = '/content/drive/MyDrive/3D_Smart/Modèles de formulations pour tous les contrats/contrats_generes/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ── Config Modèle ──────────────────────────────────────────
MODEL_REPO = "pbhappliedsystems/qwen-2.5-7B-instruct-gguf-Q4-K-M"
MODEL_FILE = "qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf"
EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with fitz.open(pdf_path) as doc:
            for page in doc:
                text += page.get_text("text") + "\n"
    except Exception as e:
        print(f"❌ Erreur lecture {pdf_path}: {e}")
    return text

class LawConsultantAgent:
    def __init__(self, rag_path):
        print('🤖 [Agent 2] Initialisation...')
        
        # 1. Chargement LLM avec fenêtre de contexte augmentée si nécessaire
        model_path = hf_hub_download(MODEL_REPO, MODEL_FILE)
        self.llm = Llama(
            model_path=model_path,
            n_ctx=4096, # Limite physique du modèle
            n_gpu_layers=-1, 
            chat_format="chatml",
            verbose=False
        )

        self.embed_model = SentenceTransformer(EMBED_MODEL_NAME)
        self.articles = []
        self.article_embeddings = None
        self._index_laws(rag_path)

    def _index_laws(self, folder):
        print(f'📂 [RAG] Analyse des lois...')
        files = list(Path(folder).glob('*.pdf'))
        
        all_chunks = []
        for f in files:
            print(f'  📖 Lecture : {f.name}')
            text = extract_text_from_pdf(str(f))
            
            # Amélioration du découpage : par mot-clé OU par bloc de texte si trop long
            # Cela évite d'avoir un seul chunk de 400k tokens
            chunks = re.split(r'(?=المادة|Article|الفصل|البند|قانون)', text)
            
            for c in chunks:
                cleaned = c.strip()
                if len(cleaned) > 2000: # Si un article est trop long, on le recoupe
                    sub_chunks = [cleaned[i:i+2000] for i in range(0, len(cleaned), 2000)]
                    all_chunks.extend(sub_chunks)
                elif len(cleaned) > 50:
                    all_chunks.append(cleaned)

        self.articles = all_chunks
        
        if self.articles:
            print(f'  ✅ {len(self.articles)} blocs indexés.')
            self.article_embeddings = self.embed_model.encode(self.articles)
        else:
            print('  ⚠️ Aucun texte extrait. Vérifiez vos PDF (OCR ?).')

    def _retrieve(self, query, k=3):
        # On réduit K à 3 pour être sûr de ne pas dépasser les 4096 tokens de contexte
        query_vec = self.embed_model.encode([query])
        sims = np.dot(self.article_embeddings, query_vec.T).flatten()
        top_idx = np.argsort(sims)[-k:][::-1]
        return [self.articles[i] for i in top_idx]

    def ask(self, question):
        context_chunks = self._retrieve(question)
        
        # Sécurité : on tronque chaque chunk pour s'assurer que le total < 3000 tokens
        # (Laissant de la place pour le système et la réponse)
        safe_context = []
        for c in context_chunks:
            safe_context.append(c[:1500]) # Limite chaque article à 1500 caractères
        
        context_str = "\n\n---\n\n".join(safe_context)

        prompt = f"""<|im_start|>system
أنت مستشار قانوني مغربي. أجب بدقة باستخدام المواد القانونية المقدمة فقط.
<|im_end|>
<|im_start|>user
السؤال: {question}
المراجع:
{context_str}
<|im_end|>
<|im_start|>assistant
"""
        print('⚙️  Analyse...')
        # Vérification finale de la taille du prompt
        tokens = self.llm.tokenize(prompt.encode('utf-8'))
        if len(tokens) > 3800:
            prompt = prompt[:2000] + "... [Texte trop long tronqué]"
            
        out = self.llm(prompt, max_tokens=1000, temperature=0.1)
        return out['choices'][0]['text'].strip()

# ── Lancement ──────────────────────────────────────────────
agent_expert = LawConsultantAgent(RAG_FOLDER)
reponse = agent_expert.ask("ما هي شروط فسخ عقد الكراء بسبب عدم أداء الإيجار؟")
print("\n" + reponse)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🤖 [Agent 2] Initialisation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📂 [RAG] Analyse des lois...
  📖 Lecture : les_lois_des_obligations_et_contrats.pdf
  ✅ 220 blocs indexés.
⚙️  Analyse...

بناءً على المواد القانونية المقدمة، يمكن تلخيص شروط فسخ عقد الكراء بسبب عدم أداء الإيجار كما يلي:

1. **التأخير في الدفع**: وفقًا للمادة 292 من القانون المدني المغربي، يمكن فسخ العقد إذا تأخر المستأجر في دفع الإيجار لمدة تزيد على ثلاثة أشهر متتالية.

2. **التأخير في الدفع لفترة طويلة**: وفقًا للمادة 299، يمكن فسخ العقد إذا تأخر المستأجر في دفع الإيجار لفترة طويلة، مع مراعاة ظروف كل حالة على حدة.

3. **التأخير في الدفع لفترة قصيرة**: وفقًا للمادة 291، يمكن فسخ العقد إذا تأخر المستأجر في دفع الإيجار لفترة قصيرة، مع مراعاة ظروف كل حالة على حدة.

4. **الإخلال بالالتزامات الأخرى**: وفقًا للمادة 299، يمكن فسخ العقد إذا أخل المستأجر بالالتزامات الأخرى المنصوص عليها في العقد، مثل الحفاظ على العقار وصيانته.

5. **الإخلال بالالتزامات الأساسية**: وفقًا للمادة 299، يمكن فسخ العقد إذا أخل المستأجر بالالتزامات الأساسية المنصوص عليها في العقد، مثل دفع الإيجار.

6. **الإخلال بالالت